# World Bank Data360: FAO_AS_4114

This notebook explores indicator `FAO_AS_4114` from the `FAO_AS` dataset using the World Bank Data360 API. Run the cells from top to bottom; the response-inspection cells are designed to reveal the API's actual JSON structure before analysis.

*Didnt end up using it

## 1. Install and Import API Client LibrariedddddffffffffffffffffffdImport the libraries used for API requests, data processing, and visualization. Install missing packages in the notebook environment only if needed.

In [46]:
# Uncomment the next line if the notebook environment does not already have these packages.
# %pip install requests pandas numpy altair

import json
from typing import Any

import altair as alt
import numpy as np
import pandas as pd
import requests

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 2. Configure the Data360 API Request

Define the page URL, dataset and indicator identifiers, headers, and optional query parameters. The exact API route can be adjusted after inspecting the page's API documentation.

In [47]:
DATA360_PAGE_URL = "https://data360.worldbank.org/en/api?indicatorid=FAO_AS_4114&datasetid=FAO_AS"
DATA360_API_ROOT = "https://data360api.worldbank.org"
DATA360_DATA_URL = f"{DATA360_API_ROOT}/data360/data"
DATA360_METADATA_URL = f"{DATA360_API_ROOT}/data360/metadata"
DATASET_ID = "FAO_AS"
INDICATOR_ID = "FAO_AS_4114"

REQUEST_HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "Data360-notebook/1.0",
}

METADATA_QUERY = {
    "query": f"&$filter=series_description/idno eq '{INDICATOR_ID}'"
}

DATA_PARAMS = {
    "DATABASE_ID": DATASET_ID,
    "INDICATOR": INDICATOR_ID,
}

PAGE_SIZE = 1_000
TIMEOUT_SECONDS = 30
print(DATA360_PAGE_URL)

https://data360.worldbank.org/en/api?indicatorid=FAO_AS_4114&datasetid=FAO_AS


## 3. Retrieve Indicator Metadata

Start with the configured request and inspect the returned metadata fields. This cell keeps the raw response available for learning the API structure.

In [48]:
metadata_response = requests.post(
    DATA360_METADATA_URL,
    json=METADATA_QUERY,
    headers=REQUEST_HEADERS,
    timeout=TIMEOUT_SECONDS,
)

print("Request URL:", metadata_response.url)
print("Status code:", metadata_response.status_code)
print("Content type:", metadata_response.headers.get("content-type"))
metadata_response.raise_for_status()
metadata_json = metadata_response.json()

if isinstance(metadata_json, dict):
    print("Top-level keys:", list(metadata_json))
else:
    print("Top-level type:", type(metadata_json).__name__)
print(json.dumps(metadata_json, indent=2)[:8_000])

Request URL: https://data360api.worldbank.org/data360/metadata
Status code: 200
Content type: application/json
Top-level keys: ['@odata.context', '@odata.count', 'value']
{
  "@odata.context": "https://itsda-dataexp-prd.search.windows.net/indexes('data360-metadata')/$metadata#docs(*)",
  "@odata.count": 1,
  "value": [
    {
      "@search.score": 1.0,
      "id": "META_FAO_AS_4114",
      "idno": "2d69de8f-1698-4bf8-b949-fc5695794401",
      "type": "indicator",
      "subtype": "timeseries",
      "disaggregation_types": [],
      "isDelete": null,
      "cfPath": null,
      "doc_type": null,
      "remove_chart_type": "stackedBar, pie",
      "data_confidentiality_code": "PU",
      "data_confidentiality_name": "Public",
      "dsd_name": "DATA360",
      "dsd_version": "1.2",
      "dsd_codelist": [
        "DATA_SOURCE",
        "COMP_BREAKDOWN_3",
        "COMP_BREAKDOWN_1",
        "OBS_CONF",
        "AGE",
        "DATABASE_ID",
        "UNIT_MEASURE",
        "SEX",
        

## 3.1. Read the Metadata in a Compact Format

This cell extracts the main metadata fields so the indicator definition can be checked before interpreting the values.

In [49]:
def find_first_dict_with_key(payload: Any, key: str) -> dict[str, Any] | None:
    if isinstance(payload, dict):
        if key in payload and isinstance(payload[key], dict):
            return payload[key]
        for value in payload.values():
            result = find_first_dict_with_key(value, key)
            if result is not None:
                return result
    elif isinstance(payload, list):
        for value in payload:
            result = find_first_dict_with_key(value, key)
            if result is not None:
                return result
    return None

series_metadata = find_first_dict_with_key(metadata_json, 'series_description') or {}
source_records = series_metadata.get('sources', [])
source_names = ', '.join(
    source.get('name', '') for source in source_records if isinstance(source, dict)
)
time_periods = series_metadata.get('time_periods', [])
period_text = ', '.join(
    f"{period.get('start')}-{period.get('end')}"
    for period in time_periods
    if isinstance(period, dict)
)

metadata_summary = pd.DataFrame({
    'Field': [
        'Indicator ID', 'Title', 'Definition provided by Data360',
        'Measurement unit', 'Periodicity', 'Time period', 'Source',
        'Downloadable metadata',
    ],
    'Value': [
        series_metadata.get('idno', INDICATOR_ID),
        series_metadata.get('title'),
        series_metadata.get('definition_long'),
        series_metadata.get('measurement_unit'),
        series_metadata.get('periodicity'),
        period_text,
        source_names,
        'https://data360files.worldbank.org/data360-data/metadata/FAO_AS/FAO_AS_4114.json',
    ],
})
display(metadata_summary)
print("\nInterpretation note: Data360 names this indicator 'safe drinking-water', but this metadata does not specify whether it means basic access or safely managed service.")

,Field,Value
0,Indicator ID,FAO_AS_4114
1,Title,NaN
2,Definition provided by Data360,Total population with access to safe drinking-...
3,Measurement unit,Percentage of population
4,Periodicity,Annual
5,Time period,1990-2023
6,Source,FAO Aquastat
7,Downloadable metadata,https://data360files.worldbank.org/data360-dat...



Interpretation note: Data360 names this indicator 'safe drinking-water', but this metadata does not specify whether it means basic access or safely managed service.


## 3.2. List All Indicators and Definitions in FAO_AS

This cell requests all indicator metadata in the `FAO_AS` dataset and displays each indicator ID with its available definition.

In [50]:
all_metadata_query = {
    'query': "&$filter=series_description/database_id eq 'FAO_AS'&$top=1000"
}
all_metadata_response = requests.post(
    DATA360_METADATA_URL,
    json=all_metadata_query,
    headers=REQUEST_HEADERS,
    timeout=TIMEOUT_SECONDS,
)
all_metadata_response.raise_for_status()
all_metadata_json = all_metadata_response.json()
all_metadata_records = all_metadata_json.get('value', [])

indicator_definitions = pd.DataFrame([
    {
        'indicator_id': record.get('series_description', {}).get('idno'),
        'definition': record.get('series_description', {}).get('definition_long')
            or record.get('series_description', {}).get('name'),
    }
    for record in all_metadata_records
    if isinstance(record, dict) and isinstance(record.get('series_description'), dict)
])
indicator_definitions = (
    indicator_definitions
    .dropna(subset=['indicator_id'])
    .drop_duplicates()
    .sort_values('indicator_id')
    .reset_index(drop=True)
)

print(f"Indicators found: {len(indicator_definitions):,}")
display(indicator_definitions)


Indicators found: 105


,indicator_id,definition
0,FAO_AS_4100,Total area of the country (excl. coastal water)
1,FAO_AS_4101,Arable land area
2,FAO_AS_4102,Permanent crops area
3,FAO_AS_4103,Cultivated area (arable land + permanent crops)
4,FAO_AS_4111,Human Development Index (HDI) [highest = 1]
...,...,...
100,FAO_AS_4553,SDG 6.4.1. Industrial Water Use Efficiency
101,FAO_AS_4554,SDG 6.4.1. Services Water Use Efficiency
102,FAO_AS_4555,% of agricultural GVA produced by irrigated ag...
103,FAO_AS_4557,Ratio between rainfed and irrigated yields


## 3.3. Discover Available Data360 Datasets

The metadata catalog contains many datasets. This cell downloads the catalog in pages and summarizes each dataset by its ID, name, and number of indicators.

In [51]:
catalog_page_size = 1_000
catalog_records = []
catalog_total = None

for catalog_skip in range(0, 100_000, catalog_page_size):
    catalog_query = {
        'query': f'&$top={catalog_page_size}&$skip={catalog_skip}',
    }
    catalog_response = requests.post(
        DATA360_METADATA_URL,
        json=catalog_query,
        headers=REQUEST_HEADERS,
        timeout=TIMEOUT_SECONDS,
    )
    catalog_response.raise_for_status()
    catalog_payload = catalog_response.json()
    catalog_total = int(catalog_payload.get('@odata.count', 0))
    catalog_page = catalog_payload.get('value', [])
    catalog_records.extend(catalog_page)
    if not catalog_page or len(catalog_records) >= catalog_total:
        break

catalog_rows = []
for record in catalog_records:
    series = record.get('series_description', {})
    if isinstance(series, dict) and series.get('database_id'):
        catalog_rows.append({
            'dataset_id': series.get('database_id'),
            'dataset_name': series.get('database_name') or series.get('database_id'),
            'indicator_id': series.get('idno'),
        })

catalog_frame = pd.DataFrame(catalog_rows)
dataset_catalog = (
    catalog_frame.groupby('dataset_id', as_index=False)
    .agg(
        dataset_name=('dataset_name', lambda values: ' | '.join(sorted(set(values)))),
        indicators=('indicator_id', 'nunique'),
        example_indicator=('indicator_id', 'first'),
    )
    .sort_values('dataset_id')
    .reset_index(drop=True)
)

print(f'Metadata records retrieved: {len(catalog_records):,} of {catalog_total:,}')
print(f'Datasets found: {len(dataset_catalog):,}')
display(dataset_catalog)


Metadata records retrieved: 12,519 of 12,519
Datasets found: 174


,dataset_id,dataset_name,indicators,example_indicator
0,BS_BTI,BS_BTI | The Bertelsmann Stiftung’s Transforma...,68,BS_BTI_Q4_2
1,BS_SGI,BS_SGI | Sustainable Governance Indicators (SGI),175,BS_SGI_133
2,DHS_SD,DHS_SD | Survey Based Indicators,1,RH_ANCC_W_IRN
3,DRMKC_INFORM,DRMKC_INFORM | INFORM Risk Index,4,INFORM_CC
4,EIA_EIAOD,EIA_EIAOD | U.S. Energy Information Administra...,5,EIA_EIAOD_GNG
...,...,...,...,...
169,WI_GRT,Green Recovery Tracker | WI_GRT,4,WI_GRT_FUNDING
170,WJP_ROL,Rule of Law Index | WJP_ROL,9,WJP_ROL_FAC_8
171,WRI_AQDT,Aqueduct 4.0 Current and Future Country Rankin...,1,WRI_AQDT_FUTURE_BWS
172,WRI_CLIMATEWATCH,Climate Watch | WRI_CLIMATEWATCH,1,WRI_CLIMATEWATCH_ALL_GHG_TRANSPORT


## 4. Fetch Indicator Data

Request the observations and display a bounded raw preview. The API response is intentionally inspected before normalization so field names can be confirmed.

In [52]:
data_request_params = {**DATA_PARAMS, "skip": 0}
data_response = requests.get(
    DATA360_DATA_URL,
    params=data_request_params,
    headers=REQUEST_HEADERS,
    timeout=TIMEOUT_SECONDS,
)

print("Request URL:", data_response.url)
print("Status code:", data_response.status_code)
print("Content type:", data_response.headers.get("content-type"))
data_response.raise_for_status()
raw_response = data_response.json()

if not isinstance(raw_response, dict) or "value" not in raw_response:
    raise ValueError("The Data360 response does not contain the expected 'value' field.")

expected_count = int(raw_response.get("count", len(raw_response["value"])))
all_records = list(raw_response["value"])

while len(all_records) < expected_count:
    page_params = {**DATA_PARAMS, "skip": len(all_records)}
    page_response = requests.get(
        DATA360_DATA_URL,
        params=page_params,
        headers=REQUEST_HEADERS,
        timeout=TIMEOUT_SECONDS,
    )
    page_response.raise_for_status()
    page_payload = page_response.json()
    page_records = page_payload.get("value", []) if isinstance(page_payload, dict) else []
    if not page_records:
        break
    all_records.extend(page_records)

raw_response["value"] = all_records
print(f"Observations retrieved: {len(all_records):,} of {expected_count:,}")
print("Top-level keys:", list(raw_response))
print(json.dumps(raw_response, indent=2)[:6_000])

Request URL: https://data360api.worldbank.org/data360/data?DATABASE_ID=FAO_AS&INDICATOR=FAO_AS_4114&skip=0
Status code: 200
Content type: application/json
Observations retrieved: 6,179 of 6,179
Top-level keys: ['count', 'value']
{
  "count": 6179,
  "value": [
    {
      "OBS_VALUE": "21.2",
      "TIME_FORMAT": "602",
      "UNIT_MULT": 0,
      "COMMENT_OBS": null,
      "OBS_STATUS": "A",
      "OBS_CONF": "PU",
      "AGG_METHOD": "_Z",
      "DECIMALS": "2",
      "COMMENT_TS": null,
      "DATA_SOURCE": null,
      "LATEST_DATA": false,
      "SEGMENT": null,
      "EV_CRITERIA": null,
      "DATABASE_ID": "FAO_AS",
      "INDICATOR": "FAO_AS_4114",
      "REF_AREA": "AFG",
      "SEX": "_Z",
      "AGE": "_Z",
      "URBANISATION": "_Z",
      "COMP_BREAKDOWN_1": "_Z",
      "COMP_BREAKDOWN_2": "_Z",
      "COMP_BREAKDOWN_3": "_Z",
      "TIME_PERIOD": "1991",
      "FREQ": "A",
      "UNIT_MEASURE": "PT_POP",
      "UNIT_TYPE": null
    },
    {
      "OBS_VALUE": "21.3",
    

## 5. Convert the API Response to a DataFrame

The helper below searches common nested response keys and accepts either a list of records or a dictionary containing records. Adjust `candidate_keys` after seeing the raw response.

In [53]:
candidate_keys = ("data", "observations", "results", "records", "items")


def find_records(payload: Any) -> list[dict[str, Any]]:
    if isinstance(payload, list):
        return [item for item in payload if isinstance(item, dict)]
    if not isinstance(payload, dict):
        return []

    for key in candidate_keys:
        if key in payload:
            records = find_records(payload[key])
            if records:
                return records

    if payload and all(isinstance(value, (str, int, float, bool, type(None))) for value in payload.values()):
        return [payload]

    for value in payload.values():
        records = find_records(value)
        if records:
            return records
    return []


records = find_records(raw_response)
indicator_df = pd.json_normalize(records)
print(f"Records found: {len(indicator_df):,}")
print("Columns:", indicator_df.columns.tolist())
indicator_df.head()

Records found: 6,179
Columns: ['OBS_VALUE', 'TIME_FORMAT', 'UNIT_MULT', 'COMMENT_OBS', 'OBS_STATUS', 'OBS_CONF', 'AGG_METHOD', 'DECIMALS', 'COMMENT_TS', 'DATA_SOURCE', 'LATEST_DATA', 'SEGMENT', 'EV_CRITERIA', 'DATABASE_ID', 'INDICATOR', 'REF_AREA', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3', 'TIME_PERIOD', 'FREQ', 'UNIT_MEASURE', 'UNIT_TYPE']


,OBS_VALUE,TIME_FORMAT,UNIT_MULT,COMMENT_OBS,OBS_STATUS,OBS_CONF,AGG_METHOD,DECIMALS,COMMENT_TS,DATA_SOURCE,...,SEX,AGE,URBANISATION,COMP_BREAKDOWN_1,COMP_BREAKDOWN_2,COMP_BREAKDOWN_3,TIME_PERIOD,FREQ,UNIT_MEASURE,UNIT_TYPE
0,21.2,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1991,A,PT_POP,None
1,21.3,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1992,A,PT_POP,None
2,21.4,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1993,A,PT_POP,None
3,21.4,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1994,A,PT_POP,None
4,21.5,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1995,A,PT_POP,None


## 6. Clean and Inspect the Dataset

Convert likely date and value fields when present, inspect missingness, remove exact duplicates, and identify country and time fields.

In [54]:
clean_df = indicator_df.copy()

possible_date_columns = [
    column for column in clean_df.columns
    if any(token in column.lower() for token in ("date", "time_period", "period"))
]
possible_value_columns = [
    column for column in ("OBS_VALUE", "VALUE", "value")
    if column in clean_df.columns
]

for column in possible_date_columns:
    if column == "TIME_PERIOD":
        clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")
    elif clean_df[column].dtype == "object":
        converted = pd.to_datetime(clean_df[column], errors="coerce")
        if converted.notna().any():
            clean_df[column] = converted

for column in possible_value_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

clean_df = clean_df.drop_duplicates()
print("Shape:", clean_df.shape)
print("Possible date/time columns:", possible_date_columns)
print("Possible value columns:", possible_value_columns)
print("Missing values:")
display(clean_df.isna().sum().sort_values(ascending=False).head(20))
clean_df.head()

Shape: (6179, 26)
Possible date/time columns: ['TIME_PERIOD']
Possible value columns: ['OBS_VALUE']
Missing values:


UNIT_TYPE           6179
COMMENT_OBS         6179
COMMENT_TS          6179
DATA_SOURCE         6179
SEGMENT             6179
EV_CRITERIA         6179
SEX                    0
UNIT_MEASURE           0
FREQ                   0
TIME_PERIOD            0
COMP_BREAKDOWN_3       0
COMP_BREAKDOWN_2       0
COMP_BREAKDOWN_1       0
URBANISATION           0
AGE                    0
OBS_VALUE              0
REF_AREA               0
INDICATOR              0
TIME_FORMAT            0
LATEST_DATA            0
dtype: int64

,OBS_VALUE,TIME_FORMAT,UNIT_MULT,COMMENT_OBS,OBS_STATUS,OBS_CONF,AGG_METHOD,DECIMALS,COMMENT_TS,DATA_SOURCE,...,SEX,AGE,URBANISATION,COMP_BREAKDOWN_1,COMP_BREAKDOWN_2,COMP_BREAKDOWN_3,TIME_PERIOD,FREQ,UNIT_MEASURE,UNIT_TYPE
0,21.2,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1991,A,PT_POP,None
1,21.3,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1992,A,PT_POP,None
2,21.4,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1993,A,PT_POP,None
3,21.4,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1994,A,PT_POP,None
4,21.5,602,0,None,A,PU,_Z,2,None,None,...,_Z,_Z,_Z,_Z,_Z,_Z,1995,A,PT_POP,None


## 7. Analyze Values by Country and Year

Use the detected country, year, and value columns when available. The summary code is intentionally conditional so it remains safe while the API schema is being explored.

In [55]:
country_candidates = ("REF_AREA", "COUNTRY", "COUNTRY_CODE", "country", "economy")
year_candidates = ("TIME_PERIOD", "YEAR", "year")

country_columns = [column for column in country_candidates if column in clean_df.columns]
year_columns = [column for column in year_candidates if column in clean_df.columns]
value_column = possible_value_columns[0] if possible_value_columns else None

print("Country columns:", country_columns)
print("Year columns:", year_columns)
print("Selected value column:", value_column)

if country_columns and value_column:
    country_summary = (
        clean_df.groupby(country_columns[0], dropna=False)[value_column]
        .agg(["count", "mean", "min", "max"])
        .sort_values("count", ascending=False)
    )
    display(country_summary.head(20))
else:
    print("A country column and numeric value column were not detected yet.")

if year_columns and value_column:
    year_summary = clean_df.groupby(year_columns[0], dropna=False)[value_column].agg(["count", "mean", "min", "max"])
    display(year_summary.head(20))
else:
    print("A year column and numeric value column were not detected yet.")

Country columns: ['REF_AREA']
Year columns: ['TIME_PERIOD']
Selected value column: OBS_VALUE


,count,mean,min,max
REF_AREA,,,,
ZWE,34,78.320588,76.9,79.6
LBY,34,71.200000,71.2,71.2
KNA,34,98.300000,98.3,98.3
SYR,34,88.488235,85.7,90.1
KWT,34,99.000000,99.0,99.0
SYC,34,95.700000,95.7,95.7
AGO,34,47.052941,45.7,49.0
SWZ,34,60.658824,38.9,74.2
LCA,34,94.941176,92.5,96.3


,count,mean,min,max
TIME_PERIOD,,,,
1990,140,77.965714,23.4,100.0
1991,143,78.165035,21.2,100.0
1992,161,79.967081,21.3,100.0
1993,168,79.613690,15.6,100.0
1994,179,79.655866,17.6,100.0
1995,181,79.991713,19.5,100.0
1996,182,80.474176,21.0,100.0
1997,183,80.830055,21.1,100.0
1998,183,81.256831,21.8,100.0


## 8. Visualize the Indicator

Create a time-series and country comparison when the response contains suitable fields. The geographic-ready summary can be exported or joined to a boundary dataset later.

In [56]:
if year_columns and value_column:
    time_plot = clean_df.groupby(year_columns[0], as_index=False)[value_column].mean()
    time_chart = (
        alt.Chart(time_plot)
        .mark_line(point=True, color='#2a9d8f')
        .encode(
            x=alt.X(f'{year_columns[0]}:Q', title='Year'),
            y=alt.Y(f'{value_column}:Q', title='Mean access (%)'),
            tooltip=[
                alt.Tooltip(f'{year_columns[0]}:O', title='Year'),
                alt.Tooltip(f'{value_column}:Q', title='Mean access (%)', format='.1f'),
            ],
        )
        .properties(width=700, height=350, title=f'Mean {value_column} over time')
    )
    display(time_chart)

if country_columns and value_column:
    top_countries = (
        clean_df.groupby(country_columns[0], as_index=False)[value_column]
        .mean()
        .sort_values(value_column, ascending=False)
        .head(15)
    )
    country_chart = (
        alt.Chart(top_countries)
        .mark_bar(color='#2a9d8f')
        .encode(
            x=alt.X(f'{value_column}:Q', title='Mean access (%)'),
            y=alt.Y(f'{country_columns[0]}:N', sort='-x', title='Economy'),
            tooltip=[
                alt.Tooltip(f'{country_columns[0]}:N', title='Economy'),
                alt.Tooltip(f'{value_column}:Q', title='Mean access (%)', format='.1f'),
            ],
        )
        .properties(width=700, height=400, title=f'Top 15 economies by mean {value_column}')
    )
    display(country_chart)

geographic_ready = (
    clean_df[[country_columns[0], value_column]].groupby(country_columns[0], as_index=False).mean()
    if country_columns and value_column
    else pd.DataFrame()
)
geographic_ready.head()

alt.Chart(...)

alt.Chart(...)

,REF_AREA,OBS_VALUE
0,AFG,40.851515
1,AGO,47.052941
2,ALB,95.576667
3,ARE,99.650000
4,ARG,97.250000


In [57]:
# Exploratory summary: coverage, latest values, and long-run change
analysis_df = clean_df[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']].copy()
analysis_df['TIME_PERIOD'] = pd.to_numeric(analysis_df['TIME_PERIOD'], errors='coerce')
analysis_df['OBS_VALUE'] = pd.to_numeric(analysis_df['OBS_VALUE'], errors='coerce')

latest_year = int(analysis_df['TIME_PERIOD'].max())
baseline_year = int(analysis_df['TIME_PERIOD'].min())
coverage = (
    analysis_df.groupby('REF_AREA')['TIME_PERIOD']
    .agg(first_year='min', last_year='max', observations='count')
    .sort_values(['last_year', 'observations'], ascending=False)
)

latest = (
    analysis_df.loc[analysis_df['TIME_PERIOD'].eq(latest_year)]
    .sort_values('OBS_VALUE', ascending=False)
)
baseline = analysis_df.loc[analysis_df['TIME_PERIOD'].eq(baseline_year), ['REF_AREA', 'OBS_VALUE']]
changes = latest[['REF_AREA', 'OBS_VALUE']].merge(
    baseline,
    on='REF_AREA',
    how='inner',
    suffixes=('_latest', '_baseline'),
)
changes['change_pp'] = changes['OBS_VALUE_latest'] - changes['OBS_VALUE_baseline']

print(f'Years available: {baseline_year}-{latest_year}')
print(f'Latest-year observations: {len(latest):,}')
print(f'Unique economies: {analysis_df["REF_AREA"].nunique():,}')
print(f'Complete {baseline_year}-{latest_year} coverage: {len(changes):,} economies')
print('\nLatest-year distribution:')
print(latest['OBS_VALUE'].describe().round(2).to_string())

print(f'\nLowest access values in {latest_year}:')
display(latest[['REF_AREA', 'OBS_VALUE']].tail(10).sort_values('OBS_VALUE'))

print(f'\nHighest access values in {latest_year}:')
display(latest[['REF_AREA', 'OBS_VALUE']].head(10))

print(f'\nLargest changes from {baseline_year} to {latest_year}:')
display(changes.sort_values('change_pp', ascending=False).head(10))

print(f'\nLargest declines from {baseline_year} to {latest_year}:')
display(changes.sort_values('change_pp').head(10))

latest_chart = (
    alt.Chart(latest.head(10))
    .mark_bar(color='#2a9d8f')
    .encode(
        x=alt.X('OBS_VALUE:Q', title='Access (%)'),
        y=alt.Y('REF_AREA:N', sort='-x', title='Economy'),
        tooltip=[
            alt.Tooltip('REF_AREA:N', title='Economy'),
            alt.Tooltip('OBS_VALUE:Q', title='Access (%)', format='.1f'),
        ],
    )
    .properties(width=360, height=300, title=f'Highest access rates, {latest_year}')
)

declines = changes.loc[changes['change_pp'] < 0].sort_values('change_pp').head(10)
decline_chart = (
    alt.Chart(declines)
    .mark_bar(color='#e76f51')
    .encode(
        x=alt.X('change_pp:Q', title='Change in percentage points'),
        y=alt.Y('REF_AREA:N', sort='x', title='Economy'),
        tooltip=[
            alt.Tooltip('REF_AREA:N', title='Economy'),
            alt.Tooltip('change_pp:Q', title='Change (pp)', format='.1f'),
        ],
    )
    .properties(width=360, height=300, title=f'Largest declines, {baseline_year}-{latest_year}')
)

display(
    alt.hconcat(latest_chart, decline_chart)
    .resolve_scale(y='independent')
    .configure_view(stroke=None)
)

Years available: 1990-2023
Latest-year observations: 188
Unique economies: 188
Complete 1990-2023 coverage: 140 economies

Latest-year distribution:
count    188.00
mean      88.03
std       15.39
min       31.70
25%       80.75
50%       95.10
75%       99.52
max      100.00

Lowest access values in 2023:


,REF_AREA,OBS_VALUE
5122,SOM,31.7
4283,PNG,40.0
1792,GNQ,47.9
130,AGO,49.0
1069,TCD,50.8
3714,MOZ,51.1
3322,MDG,51.5
1529,COD,52.4
6110,YEM,54.9
32,AFG,55.3



Highest access values in 2023:


,REF_AREA,OBS_VALUE
3912,NZL,100.0
5004,SGP,100.0
4114,NOR,100.0
5840,GBR,100.0
1396,CYP,100.0
1427,CZE,100.0
3878,NLD,100.0
1563,DNK,100.0
5202,ESP,100.0
3492,MLT,100.0



Largest changes from 1990 to 2023:


,REF_AREA,OBS_VALUE_latest,OBS_VALUE_baseline,change_pp
118,KHM,75.5,23.4,52.1
112,MLI,77.0,27.1,49.9
88,MWI,90.2,42.5,47.7
52,PRY,98.0,53.0,45.0
108,GNB,79.3,35.9,43.4
104,BFA,82.3,43.6,38.7
109,UGA,79.0,40.4,38.6
119,SWZ,74.1,38.9,35.2
59,VNM,97.6,62.7,34.9
94,GHA,88.7,55.5,33.2



Largest declines from 1990 to 2023:


,REF_AREA,OBS_VALUE_latest,OBS_VALUE_baseline,change_pp
133,YEM,54.9,66.3,-11.4
103,DZA,83.6,91.5,-7.9
131,HTI,57.7,61.9,-4.2
102,DOM,84.7,87.3,-2.6
113,ZWE,76.9,79.4,-2.5
46,NIU,98.5,99.1,-0.6
39,BGR,99.4,99.9,-0.5
34,PRK,99.7,100.0,-0.3
36,ARE,99.6,99.7,-0.1
42,KWT,99.0,99.0,0.0


alt.HConcatChart(...)

In [58]:
# Link global D1 concentration to access to safe drinking water
selected_years = [1990, 2000, 2010, 2020, 2023]
d1_people_threshold = 1_000_000

distribution_path = 'GlobalDist1000bins_1990_2026_20260922_2021_01_02_PROD.csv'
distribution = pd.read_csv(
    distribution_path,
    usecols=['year', 'code', 'welf', 'pop'],
)
distribution = distribution.loc[distribution['year'].isin(selected_years)].copy()
distribution = distribution.dropna(subset=['year', 'code', 'welf', 'pop'])
distribution['bin_pop'] = distribution['pop'] / 1000
distribution = distribution.sort_values(['year', 'welf'], kind='stable')
distribution['cum_pop'] = distribution.groupby('year')['bin_pop'].cumsum()
distribution['total_pop'] = distribution.groupby('year')['bin_pop'].transform('sum')
distribution['decile'] = np.minimum(
    np.ceil(distribution['cum_pop'] / (distribution['total_pop'] / 10)),
    10,
).astype(int)

country_population = (
    distribution.groupby(['year', 'code'], as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'country_pop_billions'})
)
country_d1 = (
    distribution.loc[distribution['decile'].eq(1)]
    .groupby(['year', 'code'], as_index=False)['bin_pop']
    .sum()
    .rename(columns={'bin_pop': 'd1_pop_billions'})
)
country_d1 = country_population.merge(country_d1, on=['year', 'code'], how='left')
country_d1['d1_pop_billions'] = country_d1['d1_pop_billions'].fillna(0)
country_d1['d1_pop_people'] = country_d1['d1_pop_billions'] * 1e9
country_d1['d1_population_share'] = 100 * country_d1['d1_pop_billions'] / country_d1['country_pop_billions']
country_d1['above_threshold'] = country_d1['d1_pop_people'] > d1_people_threshold

water = analysis_df.rename(columns={'REF_AREA': 'code', 'TIME_PERIOD': 'year', 'OBS_VALUE': 'water_access_pct'})
water = water[['code', 'year', 'water_access_pct']].loc[water['year'].isin(selected_years)]
water_d1 = country_d1.merge(water, on=['code', 'year'], how='inner')

threshold_counts = (
    water_d1.groupby('year')['above_threshold']
    .sum()
    .astype(int)
    .rename('countries_above_threshold')
    .reset_index()
)
print(f'People threshold: {d1_people_threshold:,}')
print('Countries above the threshold by year:')
display(threshold_counts)

latest_link = water_d1.loc[
    (water_d1['year'].eq(2023)) & water_d1['above_threshold'],
    ['code', 'd1_pop_people', 'd1_population_share', 'water_access_pct'],
].sort_values('d1_pop_people', ascending=False)
print('Countries with more than the threshold of people in global D1, 2023:')
display(latest_link.head(20))

plot_data = water_d1.loc[
    water_d1['year'].eq(2023) & water_d1['above_threshold']
].copy()
label_data = plot_data.nlargest(8, 'd1_pop_people')
points = (
    alt.Chart(plot_data)
    .mark_circle(color='#d95f02', opacity=0.75, stroke='white')
    .encode(
        x=alt.X('water_access_pct:Q', title='Population with access to safe drinking water (%)', scale=alt.Scale(domain=[0, 105])),
        y=alt.Y('d1_population_share:Q', title='Share of country population in global D1 (%)'),
        size=alt.Size('d1_pop_people:Q', title='People in D1', scale=alt.Scale(range=[40, 700])),
        tooltip=[
            alt.Tooltip('code:N', title='Economy'),
            alt.Tooltip('d1_pop_people:Q', title='People in D1', format=',.0f'),
            alt.Tooltip('d1_population_share:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('water_access_pct:Q', title='Water access (%)', format='.1f'),
        ],
    )
)
labels = (
    alt.Chart(label_data)
    .mark_text(align='left', dx=5, fontSize=11)
    .encode(
        x='water_access_pct:Q',
        y='d1_population_share:Q',
        text='code:N',
    )
)
d1_water_chart = (
    (points + labels)
    .properties(
        width=850,
        height=500,
        title='Countries with many people in global D1, 2023',
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(d1_water_chart)

People threshold: 1,000,000
Countries above the threshold by year:


,year,countries_above_threshold
0,1990,37
1,2000,52
2,2010,55
3,2020,59
4,2023,60


Countries with more than the threshold of people in global D1, 2023:


,code,d1_pop_people,d1_population_share,water_access_pct
810,NGA,9.160894e+07,40.2,68.5
723,COD,8.685337e+07,82.1,52.4
764,IND,4.889437e+07,3.4,94.1
818,PAK,4.851088e+07,19.6,91.4
743,ETH,4.246826e+07,33.0,57.3
861,TZA,3.111042e+07,46.7,55.6
872,YEM,2.717965e+07,69.0,54.9
803,MOZ,2.704267e+07,80.4,51.1
862,UGA,2.578800e+07,53.0,79.0
775,KEN,2.368509e+07,42.8,63.2


alt.LayerChart(...)

In [59]:
# Test the left-behind hypothesis for the water dimension
water_population = country_population.merge(
    water,
    on=['year', 'code'],
    how='inner',
)
water_population['people_without_access_billions'] = (
    water_population['country_pop_billions']
    * (1 - water_population['water_access_pct'] / 100)
)

global_water_summary = (
    water_population.groupby('year')
    .apply(
        lambda group: pd.Series({
            'world_population_billions': group['country_pop_billions'].sum(),
            'weighted_access_pct': (
                group['country_pop_billions'].mul(group['water_access_pct']).sum()
                / group['country_pop_billions'].sum()
            ),
            'people_without_access_billions': group['people_without_access_billions'].sum(),
        }),
        include_groups=False,
    )
    .reset_index()
)
global_water_summary['without_access_pct'] = (
    100
    * global_water_summary['people_without_access_billions']
    / global_water_summary['world_population_billions']
)
global_water_summary['times_10pct_reference'] = (
    global_water_summary['without_access_pct'] / 10
)

display(global_water_summary.round(2))

latest_water = global_water_summary.loc[global_water_summary['year'].eq(2023)].iloc[0]
print(
    f"In 2023, {latest_water['without_access_pct']:.1f}% of the represented population "
    f"was without access to safe drinking water."
)
print(
    f"That is {latest_water['times_10pct_reference']:.1f} times the 10% reference group."
)

water_line = (
    alt.Chart(global_water_summary)
    .mark_line(point=True, color='#d95f02', strokeWidth=2.5)
    .encode(
        x=alt.X('year:Q', title='Year', axis=alt.Axis(values=selected_years, format='d')),
        y=alt.Y('without_access_pct:Q', title='Population without access (%)'),
        tooltip=[
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('without_access_pct:Q', title='Without access (%)', format='.1f'),
            alt.Tooltip('weighted_access_pct:Q', title='Weighted access (%)', format='.1f'),
        ],
    )
)
water_reference = (
    alt.Chart(pd.DataFrame({'reference_pct': [10]}))
    .mark_rule(color='#222222', strokeDash=[6, 4], strokeWidth=1.5)
    .encode(y=alt.Y('reference_pct:Q', title='Population without access (%)'))
)
water_gap_chart = (
    (water_line + water_reference)
    .properties(
        width=850,
        height=400,
        title='The water-access gap is close to, but not below, 10% in 2023',
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
)
display(water_gap_chart)

,year,world_population_billions,weighted_access_pct,people_without_access_billions,without_access_pct,times_10pct_reference
0,1990,4.77,75.82,1.15,24.18,2.42
1,2000,6.08,82.50,1.06,17.50,1.75
2,2010,6.92,88.43,0.80,11.57,1.16
3,2020,7.82,90.01,0.78,9.99,1.00
4,2023,8.02,89.73,0.82,10.27,1.03


In 2023, 10.3% of the represented population was without access to safe drinking water.
That is 1.0 times the 10% reference group.


alt.LayerChart(...)

## 10. Correlation Between Global D1 Concentration and Water Access

This exploratory analysis tests whether countries with a larger share of their population in the global poorest decile also tend to have lower access to safe drinking water. The correlations describe association, not causation. The global D1 measure is not the same as each country's national poorest 10%.

In [61]:
correlation_data = water_d1[
    ['year', 'code', 'd1_population_share', 'd1_pop_people', 'water_access_pct']
].dropna()

correlation_results = (
    correlation_data.groupby('year')
    .apply(
        lambda group: pd.Series({
            'pearson_r': group['d1_population_share'].corr(
                group['water_access_pct'],
                method='pearson',
            ),
            'spearman_r': group['d1_population_share'].rank().corr(
                group['water_access_pct'].rank(),
                method='pearson',
            ),
            'observations': len(group),
        }),
        include_groups=False,
    )
    .reset_index()
    .sort_values('year')
)

print('Correlation between the share of a country in global D1 and water access:')
display(correlation_results.round(3))

Correlation between the share of a country in global D1 and water access:


,year,pearson_r,spearman_r,observations
0,1990,-0.670,-0.711,138.0
1,2000,-0.684,-0.767,182.0
2,2010,-0.679,-0.749,184.0
3,2020,-0.724,-0.749,186.0
4,2023,-0.738,-0.746,186.0


## 11. Scatter Plot: Global D1 Share and Water Access

The scatter plot below shows the latest available year. Each point is a country; the line summarizes the overall linear association in that year.

In [62]:
scatter_year = int(correlation_data['year'].max())
scatter_data = correlation_data.loc[
    correlation_data['year'].eq(scatter_year)
].copy()

scatter_points = (
    alt.Chart(scatter_data)
    .mark_circle(size=75, opacity=0.75, color='#2a9d8f', stroke='white')
    .encode(
        x=alt.X(
            'd1_population_share:Q',
            title='Population share in global D1 (%)',
            scale=alt.Scale(zero=False),
        ),
        y=alt.Y(
            'water_access_pct:Q',
            title='Population with access to water (%)',
            scale=alt.Scale(domain=[0, 105]),
        ),
        tooltip=[
            alt.Tooltip('code:N', title='Economy'),
            alt.Tooltip('d1_population_share:Q', title='Global D1 share (%)', format='.1f'),
            alt.Tooltip('water_access_pct:Q', title='Water access (%)', format='.1f'),
        ],
    )
    .properties(
        width=750,
        height=450,
        title=f'Global D1 share and water access, {scatter_year}',
    )
)

scatter_trend = (
    alt.Chart(scatter_data)
    .transform_regression(
        'd1_population_share',
        'water_access_pct',
    )
    .mark_line(color='#e76f51', strokeWidth=2.5)
    .encode(
        x='d1_population_share:Q',
        y='water_access_pct:Q',
    )
)

display(
    (scatter_points + scatter_trend)
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
    .resolve_scale(x='independent', y='independent')
 )

alt.LayerChart(...)

## 12. Scatter Plot: Global D1 Share and Water Access in 1990

The same relationship is shown for 1990 to compare the starting point with the latest available year.

In [63]:
scatter_year_1990 = 1990
scatter_data_1990 = correlation_data.loc[
    correlation_data['year'].eq(scatter_year_1990)
].copy()

scatter_points_1990 = (
    alt.Chart(scatter_data_1990)
    .mark_circle(size=75, opacity=0.75, color='#2a9d8f', stroke='white')
    .encode(
        x=alt.X(
            'd1_population_share:Q',
            title='Population share in global D1 (%)',
            scale=alt.Scale(zero=False),
        ),
        y=alt.Y(
            'water_access_pct:Q',
            title='Population with access to water (%)',
            scale=alt.Scale(domain=[0, 105]),
        ),
        tooltip=[
            alt.Tooltip('code:N', title='Economy'),
            alt.Tooltip('d1_population_share:Q', title='Global D1 share (%)', format='.1f'),
            alt.Tooltip('water_access_pct:Q', title='Water access (%)', format='.1f'),
        ],
    )
    .properties(
        width=750,
        height=450,
        title='Global D1 share and water access, 1990',
    )
)

scatter_trend_1990 = (
    alt.Chart(scatter_data_1990)
    .transform_regression(
        'd1_population_share',
        'water_access_pct',
    )
    .mark_line(color='#e76f51', strokeWidth=2.5)
    .encode(
        x='d1_population_share:Q',
        y='water_access_pct:Q',
    )
)

display(
    (scatter_points_1990 + scatter_trend_1990)
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridColor='#d9e2ec', gridDash=[3, 3])
    .resolve_scale(x='independent', y='independent')
 )

alt.LayerChart(...)

## Scatter plot: water access and global D1

Each bubble is a country. The horizontal axis is the percentage of that country's population in the global D1, the vertical axis is national access to safe drinking water, and bubble size reflects the population weight. The two panels show the first and last common years.

In [12]:
from pathlib import Path
import sys
import importlib
import altair as alt

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'indicator_analysis_helpers.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
import indicator_analysis_helpers
importlib.reload(indicator_analysis_helpers)
from indicator_analysis_helpers import REGION_COLORS, run_indicator_analysis

water_result = run_indicator_analysis('FAO_AS', 'FAO_AS_4114', 'coverage_pct')
plot_df = water_result['joined'][water_result['joined']['year'].isin([water_result['first_year'], water_result['last_year']])].copy()
plot_df = plot_df.dropna(subset=['region'])
plot_df['year'] = plot_df['year'].astype(str)
plot_years = [str(water_result['first_year']), str(water_result['last_year'])]

In [11]:
point_chart = (
    alt.Chart(plot_df)
    .mark_circle(opacity=0.7, stroke='white', strokeWidth=0.5)
    .encode(
        x=alt.X('d1_country_population_pct:Q', title='Population in global D1 (%)'),
        y=alt.Y('indicator_value:Q', title='Population with access to safe drinking water (%)', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('region:N', title='World Bank region', scale=alt.Scale(domain=list(REGION_COLORS), range=list(REGION_COLORS.values()))),
        size=alt.Size('country_population:Q', legend=None, scale=alt.Scale(range=[40, 900])),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('region:N', title='Region'),
            alt.Tooltip('year:O', title='Year'),
            alt.Tooltip('d1_country_population_pct:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('indicator_value:Q', title='Water access (%)', format='.1f'),
        ],
    )
    .properties(width=420, height=420)
)
water_scatter = point_chart.facet(column=alt.Column('year:O', title=None, sort=plot_years)).properties(title='Water access versus population in the global D1')
water_scatter

alt.FacetChart(...)

## Where the poorest decile lives

This measure only counts **where people live**, so it makes no assumption about who within a country has access.

For each year it reports the share of the global poorest decile living in countries where fewer than half the population has safe drinking water, next to the same share for the world population. The ratio between the two lines is the gap.

In [13]:
from indicator_analysis_helpers import low_coverage_chart

low_coverage = water_result['low_coverage']
display(low_coverage.iloc[[0, -1]].round(2))
print(f"Gap ratio: {low_coverage.iloc[0]['gap_ratio']:.2f}x in {low_coverage.iloc[0]['year']:.0f}"
      f" -> {low_coverage.iloc[-1]['gap_ratio']:.2f}x in {low_coverage.iloc[-1]['year']:.0f}")
low_coverage_chart(low_coverage, 'Safe drinking water')

,year,d1_share_below,world_share_below,gap_ratio,countries_below,countries_with_data
0,1990,14.82,5.96,2.49,21,138
33,2023,3.90,0.84,4.65,4,186


Gap ratio: 2.49x in 1990 -> 4.65x in 2023


alt.Chart(...)